In [1]:
import sys, site

user_site = site.getusersitepackages()
sys.path = [p for p in sys.path if p != user_site]
print("Removed user site:", user_site)
print("Using python:", sys.executable)

Removed user site: /home/sjrao/.local/lib/python3.12/site-packages
Using python: /opt/conda/envs/cse234/bin/python


In [2]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
from datasets import Dataset
import itertools
import json

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 512

### Load Dataset and Specify Train and Eval Partitions

In [4]:
import json

with open("train.json", "r") as f:
    train_data = json.load(f)
    train_dataset = Dataset.from_list(train_data)

with open("validation.json", "r") as f:
    validation_data = json.load(f)
    validation_dataset = Dataset.from_list(validation_data)

### Define Data Processing Function

In [5]:

def basic_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        table_name = schema_data["table_names_original"][i]
        schema[table_name].append(name)

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema: {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "completion": [{"role": "assistant", "content": answer}]
    }

def pkfk_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    column_info = schema_data["column_names_original"]
    primary_keys = schema_data.get("primary_keys", [])
    foreign_keys = schema_data.get("foreign_keys", [])
    col_annotations = {}

    for pk in primary_keys:
        if isinstance(pk, list):
            for col_idx in pk:
                col_annotations[col_idx] = "(PK)"
        else:
            col_annotations[pk] = "(PK)"

    for from_idx, to_idx in foreign_keys:
        to_table_idx = column_info[to_idx][0]
        to_table_name = schema_data["table_names_original"][to_table_idx]
        if from_idx in col_annotations and col_annotations[from_idx] == "(PK)":
            col_annotations[from_idx] = f"(PK,FK→{to_table_name})"
        else:
            col_annotations[from_idx] = f"(FK→{to_table_name})"

    schema = {}
    for col_idx, (table_idx, col_name) in enumerate(column_info):
        if table_idx == -1:
            continue
        table_name = schema_data["table_names_original"][table_idx]
        if table_name not in schema:
            schema[table_name] = []
        annotation = col_annotations.get(col_idx, "")
        if annotation:
            annotated_col = f"{col_name} {annotation}"
        else:
            annotated_col = col_name
        schema[table_name].append(annotated_col)

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema with PK/FK annotations, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists (without annotations in the output)."
    )
    prompt = (
        f"Database schema (PK/FK annotated): {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only — column names without annotations: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "completion": [{"role": "assistant", "content": answer}]
    }

def sorted_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        table_name = schema_data["table_names_original"][i]
        schema[table_name].append(name)

    sorted_schema = {}
    for table_name in sorted(schema.keys()):
        sorted_schema[table_name] = sorted(schema[table_name])

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema (sorted): {sorted_schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        "completion": [{"role": "assistant", "content": answer}]
    }

### Initialize Experiment

In [6]:
# Every experiment instance must be uniquely named
experiment = Experiment(experiment_name="experkrj", mode="fit")

Experiment experkrj is currently running. Returning the same experiment object.


### Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [7]:
import torch

TINYLLAMA = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
QWEN = "Qwen/Qwen2.5-1.5B-Instruct"

# Keep both models for comparison once stability is fixed.
MODEL_CONFIGS = [
    {"name": TINYLLAMA, "lr": 1e-3},
    {"name": QWEN, "lr": 1e-4},
]

# Keep LoRA only for stability; re-enable QLoRA after allocator issues are gone.
ADAPTER_TYPES = ["lora"]
RANKS = [8]
TARGET_MODULES_OPTIONS = [
    ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
]
PROMPT_FUNCS = [
    ("basic", basic_formatting_function),
    ("pkfk", pkfk_formatting_function),
    ("sorted", sorted_formatting_function),
]

all_configs = []
for model_cfg, adapter_type, r, target_modules, (prompt_name, fmt_func) in itertools.product(
    MODEL_CONFIGS, ADAPTER_TYPES, RANKS, TARGET_MODULES_OPTIONS, PROMPT_FUNCS
):
    model_kwargs = {
        "torch_dtype": torch.float16,
        "use_cache": False,
    }

    all_configs.append(RFModelConfig(
        model_name=model_cfg["name"],
        peft_config=RFLoraConfig(
            r=r,
            lora_alpha=r * 2,
            lora_dropout=0.1,
            target_modules=target_modules,
            bias="none",
        ),
        training_args=RFSFTConfig(
            learning_rate=model_cfg["lr"],
            lr_scheduler_type="linear",
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            max_steps=96,
            gradient_accumulation_steps=2,
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=8,
            bf16=False,
            fp16=True,
        ),
        model_type="causal_lm",
        model_kwargs=model_kwargs,
        formatting_func=fmt_func,
    ))

print(f"Total configs: {len(all_configs)}")
config_set = List(all_configs)

Total configs: 6


#### Define Model Creation Function for All Model Types Across Configs

In [8]:
def sample_create_model(model_config):
    """Creates model + tokenizer with conservative CUDA settings for worker stability."""
    import gc
    import os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch

    # Reduce allocator fragmentation across repeated worker model loads.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_name = model_config["model_name"]
    model_kwargs = dict(model_config["model_kwargs"])
    # Avoid HF auto-placement during load; let trainer/accelerate handle placement.
    model_kwargs.pop("device_map", None)
    model_kwargs.setdefault("low_cpu_mem_usage", True)
    model_kwargs.setdefault("attn_implementation", "eager")

    use_4bit = model_kwargs.pop("use_4bit", False)
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model_kwargs["quantization_config"] = bnb_config

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return (model, tokenizer)

#### Generate Config Group

In [ ]:
config_group = RFGridSearch(
    configs=config_set,
    trainer_type="SFT"
)

### Run Multi-Config Training

In [ ]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    validation_dataset,
    num_chunks=1,
    seed=42,
    # max_concurrent_runs=1,
 )

INFO 05-29 22:20:51 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-29 22:21:07 [__init__.py:216] Automatically detected platform cuda.


### End Current Experiment

In [11]:
experiment.end()

Experiment experkrj ended
Workers stopped
